# 02 — Validation Suite: Drift (PSI/KS), Sensitivity, Stress, LGD/EAD (Proxy)

## Goal
Run Model Risk validation tests on the Champion model:
- segment performance (business relevance)
- stability & drift monitoring (PSI for features and score; KS score shift)
- sensitivity analysis (±5/±10 shocks + ranking stability)
- stress scenarios and Expected Loss proxy via PD × LGD × EAD

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    DATE_COL, STATUS_COL, TARGET_COL,
    NUM_FEATURES, CAT_FEATURES, ENGINEERED_FEATURES,
)

from src.data.load_raw import load_raw_loans
from src.data.clean import filter_closed_loans_and_target
from src.data.features import add_credit_history_length
from src.data.split import time_split

from src.models.champion_logit import fit_champion, predict_pd

from src.validation.metrics import performance_report
from src.validation.stability import compute_psi_table, score_ks_2sample
from src.validation.sensitivity import sensitivity_shocks
from src.validation.stress import stress_test

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## Data preparation and Champion fit

We replicate the same pipeline used previously:
- load controlled columns (memory-safe, governance-friendly)
- closed-outcome PD target only
- underwriting-safe feature engineering
- true out-of-time split (OOT)

Then we fit the Champion (logistic) on Train and generate predicted PDs for Train and OOT.

In [2]:
df = load_raw_loans()
df = filter_closed_loans_and_target(df)
df = add_credit_history_length(df)

train, oot = time_split(df)

pipe, kept_num, kept_cat, dropped = fit_champion(train)

# Add predictions
train = train.copy()
oot = oot.copy()
train["pd_pred"] = predict_pd(pipe, train, kept_num, kept_cat)
oot["pd_pred"]   = predict_pd(pipe, oot, kept_num, kept_cat)

print("Train:", train.shape, "default rate:", train[TARGET_COL].mean())
print("OOT  :", oot.shape,   "default rate:", oot[TARGET_COL].mean())

print("\nKept numeric:", kept_num)
print("Kept categorical:", kept_cat)
print("\nDropped numeric:", dropped["dropped_num"])
print("Dropped categorical:", dropped["dropped_cat"])

Train: (1163294, 32) default rate: 0.20667260383015815
OOT  : (47191, 32) default rate: 0.14729503507024644

Kept numeric: ['annual_inc', 'dti', 'int_rate', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'installment', 'funded_amnt', 'loan_amnt', 'credit_history_length_years']
Kept categorical: ['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'application_type']

Dropped numeric: []
Dropped categorical: []


## Segment performance (business relevance)

We evaluate performance by:
- `grade`
- `term`
- `purpose`

Why:
- stakeholders care about where the model works/fails
- segment degradation can reveal portfolio drift
- high-volume segments drive business impact

In [3]:
def segment_report(df_seg: pd.DataFrame, segment_col: str, min_n: int = 500) -> pd.DataFrame:
    rows = []
    for seg_value, g in df_seg.groupby(segment_col):
        if len(g) < min_n:
            continue
        rep = performance_report(g[TARGET_COL].values, g["pd_pred"].values)
        rows.append({"segment": seg_value, "count": len(g), **rep})
    return pd.DataFrame(rows).sort_values("count", ascending=False)

for col in ["grade", "term", "purpose"]:
    print(f"\n=== {col} (Train) ===")
    display(segment_report(train, col).head(15))

    print(f"\n=== {col} (OOT) ===")
    display(segment_report(oot, col).head(15))


=== grade (Train) ===


,segment,count,AUC,PR_AUC,Gini,KS,Brier,Top10pct_Default_Capture,Mean_PD,Obs_Default_Rate
2,C,337586,0.601575,0.306808,0.203150,0.143566,0.172851,0.157638,0.230057,0.229873
1,B,336801,0.603168,0.190202,0.206335,0.148638,0.115719,0.165154,0.135980,0.136128
0,A,194669,0.626189,0.095258,0.252377,0.183259,0.056547,0.186456,0.060941,0.060914
3,D,174460,0.605966,0.406677,0.211932,0.148267,0.208479,0.153242,0.312861,0.313115
4,E,82954,0.608022,0.494367,0.216044,0.155628,0.231280,0.143637,0.397720,0.397305
5,F,28571,0.602550,0.553776,0.205100,0.148408,0.240692,0.132612,0.465657,0.465577
6,G,8253,0.576550,0.579425,0.153101,0.122101,0.245830,0.122782,0.513252,0.512177



=== grade (OOT) ===


,segment,count,AUC,PR_AUC,Gini,KS,Brier,Top10pct_Default_Capture,Mean_PD,Obs_Default_Rate
1,B,12815,0.607103,0.147530,0.214207,0.160112,0.097860,0.145506,0.138506,0.109403
2,C,12365,0.609538,0.226293,0.219076,0.165241,0.143751,0.148966,0.228015,0.172099
0,A,10590,0.638583,0.081809,0.277165,0.207110,0.046556,0.192678,0.060430,0.049008
3,D,8180,0.585193,0.289567,0.170385,0.127852,0.186158,0.133739,0.305775,0.241320
4,E,2577,0.623089,0.373028,0.246178,0.175643,0.201900,0.168317,0.370259,0.274350
5,F,545,0.563351,0.354007,0.126703,0.136139,0.235672,0.091429,0.451460,0.321101



=== term (Train) ===


...

,segment,count,AUC,PR_AUC,Gini,KS,Brier,Top10pct_Default_Capture,Mean_PD,Obs_Default_Rate
0,36 months,34371,0.703224,0.235425,0.406447,0.302256,0.103695,0.231130,0.150752,0.124116
1,60 months,12820,0.643412,0.304294,0.286824,0.214855,0.168550,0.174674,0.297498,0.209438



=== purpose (Train) ===


,segment,count,AUC,PR_AUC,Gini,KS,Brier,Top10pct_Default_Capture,Mean_PD,Obs_Default_Rate
2,debt_consolidation,684140,0.708085,0.395196,0.416169,0.301235,0.154410,0.220926,0.218195,0.218229
1,credit_card,260797,0.718583,0.348763,0.437167,0.319394,0.131226,0.242783,0.175406,0.175324
3,home_improvement,74789,0.706494,0.341715,0.412989,0.301081,0.138661,0.223031,0.185460,0.185428
8,other,63296,0.685724,0.375497,0.371448,0.265832,0.157933,0.211387,0.219412,0.219208
5,major_purchase,23254,0.725678,0.395152,0.451357,0.329430,0.142117,0.248595,0.199142,0.198934
6,medical,12792,0.669416,0.375057,0.338831,0.248256,0.163550,0.199103,0.227929,0.226548
10,small_business,11303,0.649365,0.445483,0.298731,0.215489,0.201008,0.170011,0.308277,0.308591
0,car,10988,0.717881,0.325120,0.435761,0.325656,0.121496,0.245392,0.155296,0.157991
7,moving,7763,0.659251,0.392093,0.318502,0.232766,0.173980,0.193312,0.247869,0.246554
11,vacation,7554,0.660130,0.329428,0.320260,0.231697,0.150056,0.201203,0.197300,0.198041



=== purpose (OOT) ===


,segment,count,AUC,PR_AUC,Gini,KS,Brier,Top10pct_Default_Capture,Mean_PD,Obs_Default_Rate
2,debt_consolidation,23983,0.691533,0.265723,0.383066,0.287048,0.124903,0.203986,0.204474,0.150648
1,credit_card,9409,0.685213,0.203017,0.370426,0.274539,0.102724,0.222852,0.176482,0.112552
7,other,4587,0.712195,0.310324,0.424390,0.326965,0.131950,0.208861,0.178811,0.172226
3,home_improvement,3830,0.711925,0.237205,0.423850,0.327995,0.110117,0.218876,0.166999,0.130026
5,major_purchase,1526,0.674872,0.309125,0.349745,0.263687,0.146435,0.195876,0.183023,0.190695
6,medical,907,0.712896,0.344584,0.425792,0.337281,0.148505,0.191257,0.198151,0.201764
4,house,864,0.706905,0.301926,0.413810,0.338160,0.127211,0.212766,0.149948,0.163194
0,car,681,0.750370,0.292977,0.500739,0.373401,0.091291,0.276316,0.141235,0.111601


**Interpretation checklist**
- Focus on high-count segments first.
- Look for:
  - degradation in AUC/KS from Train → OOT
  - large gaps between Mean_PD and observed default rate within a segment (calibration drift)
- Any concentrated degradation becomes a monitoring and remediation recommendation.

**Segment findings (summary)**
- Performance is broadly consistent across high-volume grades (A/B/C), with moderate discrimination.
- OOT shows reduced presence of extreme risk grades (F/G), suggesting **portfolio mix drift**.
- `term=60 months` appears weaker OOT than `36 months` (lower AUC/KS), indicating a segment to monitor.
- For `purpose`, conclusions should focus on high-count categories; low-count purposes are noisy.

## Drift monitoring: PSI (features + score)

PSI measures how much a distribution changed from Train to OOT.
We compute PSI for:
- each kept model input feature
- the model score (`pd_pred`)

Traffic-light thresholds (rule of thumb):
- PSI < 0.10: GREEN (stable)
- 0.10–0.25: AMBER (moderate drift)
- *>* 0.25: RED (severe drift)

In [13]:
psi_table = compute_psi_table(train, oot, kept_num, kept_cat, score_col="pd_pred")
psi_table

,feature,type,psi,flag
8,revol_util,numeric,0.290233,RED
21,application_type,categorical,0.245669,AMBER
2,int_rate,numeric,0.127119,AMBER
7,revol_bal,numeric,0.088099,GREEN
20,purpose,categorical,0.078296,GREEN
16,sub_grade,categorical,0.075582,GREEN
11,funded_amnt,numeric,0.062500,GREEN
12,loan_amnt,numeric,0.062500,GREEN
19,verification_status,categorical,0.045614,GREEN
1,dti,numeric,0.043236,GREEN


In [14]:
display(psi_table.sort_values("psi", ascending=False).head(15))

score_psi = float(psi_table.loc[psi_table["feature"] == "pd_pred", "psi"].iloc[0])
print("Score PSI (pd_pred):", score_psi)

,feature,type,psi,flag
8,revol_util,numeric,0.290233,RED
21,application_type,categorical,0.245669,AMBER
2,int_rate,numeric,0.127119,AMBER
7,revol_bal,numeric,0.088099,GREEN
20,purpose,categorical,0.078296,GREEN
16,sub_grade,categorical,0.075582,GREEN
11,funded_amnt,numeric,0.062500,GREEN
12,loan_amnt,numeric,0.062500,GREEN
19,verification_status,categorical,0.045614,GREEN
1,dti,numeric,0.043236,GREEN


Score PSI (pd_pred): 0.025876937086374445


**How to use PSI results**
- RED/AMBER features are primary candidates for:
  - deeper investigation (data sourcing, portfolio mix, policy changes)
  - monitoring thresholds in production
- Score PSI is especially important:
  - high score PSI can indicate the model's output distribution shifted materially

**Drift drivers (PSI)**
- `revol_util` is the main drift driver (RED), indicating a material shift in revolving utilization between Train and OOT.
- `application_type` shows high drift (AMBER close to RED), consistent with changing share of individual vs joint applications over time.
- `int_rate` drift (AMBER) suggests changes in pricing / risk appetite / credit cycle conditions.

These variables should be prioritized for monitoring and investigation.

## Score distribution shift: KS two-sample test

We complement PSI with a two-sample KS test on `pd_pred` (Train vs OOT).

With large sample sizes, p-values can be tiny even for small shifts.
We focus on the **effect size** (KS statistic) and PSI magnitudes.

In [6]:
ks_shift = score_ks_2sample(train["pd_pred"].values, oot["pd_pred"].values)
ks_shift

{'ks_stat': 0.057604866912486674, 'p_value': 2.986696675891732e-131}

**Score shift (KS)**

The two-sample KS statistic (~0.058) indicates a measurable but not extreme shift in the score distribution from Train to OOT. Given the large sample size, the p-value is not decision-useful; we focus on the KS effect size and PSI magnitudes.

## Sensitivity analysis: small shocks + ranking stability

We perturb key numeric drivers by ±5% and ±10% and measure:
- change in mean predicted PD
- change in tail PD (p95)
- Spearman rank correlation (does risk ordering change?)

Why:
- input uncertainty exists (reporting errors, small shifts)
- a robust model should not be extremely sensitive to small perturbations

In [7]:
# Define predict function compatible with sensitivity module
predict_fn = lambda d: predict_pd(pipe, d, kept_num, kept_cat)

drivers = [c for c in ["annual_inc", "dti", "revol_util"] if c in kept_num]
drivers

['annual_inc', 'dti', 'revol_util']

In [8]:
sens_tables = []
for f in drivers:
    sens_tables.append(sensitivity_shocks(oot, predict_fn, feature=f))
sens_table = pd.concat(sens_tables, ignore_index=True) if sens_tables else pd.DataFrame()
sens_table

,feature,shock,mean_pd_base,mean_pd_new,delta_mean_pd,p95_base,p95_new,delta_p95,spearman_rank_corr
0,annual_inc,-0.10,0.190617,0.192223,0.001606,0.417995,0.420332,0.002337,0.999958
1,annual_inc,-0.05,0.190617,0.191418,0.000801,0.417995,0.419096,0.001101,0.999989
2,annual_inc,0.05,0.190617,0.189820,-0.000797,0.417995,0.416774,-0.001222,0.999990
3,annual_inc,0.10,0.190617,0.189027,-0.001591,0.417995,0.415630,-0.002365,0.999959
4,dti,-0.10,0.190617,0.186945,-0.003673,0.417995,0.410241,-0.007754,0.999867
5,dti,-0.05,0.190617,0.188775,-0.001842,0.417995,0.414109,-0.003886,0.999967
6,dti,0.05,0.190617,0.192470,0.001853,0.417995,0.422023,0.004028,0.999968
7,dti,0.10,0.190617,0.194335,0.003717,0.417995,0.425906,0.007911,0.999872
8,revol_util,-0.10,0.190617,0.190006,-0.000611,0.417995,0.416765,-0.001231,0.999995
9,revol_util,-0.05,0.190617,0.190311,-0.000306,0.417995,0.417401,-0.000595,0.999999


**Interpretation**
- Large `delta_mean_pd` or `delta_p95` indicates high sensitivity.
- Spearman correlation close to 1 indicates stable ranking (operationally desirable).
- Sensitivity findings feed into a “model vulnerability statement” in the final report.

**Sensitivity findings**
- Ranking is highly stable under ±5%/±10% shocks (Spearman ~ 1), which is operationally desirable.
- Among tested drivers, `dti` shows the largest impact on mean PD and tail PD (p95), making it the primary sensitivity driver.
- Notably, `revol_util` shows strong drift (PSI RED) but relatively small local sensitivity to small shocks; drift risk and sensitivity risk are distinct.

## Stress scenarios + Expected Loss proxy (PD × LGD × EAD)

We run two stylized stresses on OOT:
- Mild: income down, DTI up
- Severe: larger income down, larger DTI up, utilization up

We compute:
- PD mean and p95 shift
- EL proxy shift, using:
  - EAD proxy from funded amount and principal repaid
  - LGD proxy from recoveries over exposure (defaults only)

This is not a regulatory IFRS9 model, but demonstrates end-to-end credit risk reasoning.

In [9]:
stress_table, oot_with_proxies = stress_test(oot, predict_fn, default_col=TARGET_COL)
stress_table

,scenario,mean_pd,delta_mean_pd,p95_pd,delta_p95,EL_proxy_mean,delta_EL_proxy,LGD_avg_defaults
0,mild,0.195145,0.004528,0.427175,0.009180,647.706794,13.718138,0.979338
1,severe,0.200342,0.009725,0.436968,0.018973,663.375656,29.387000,0.979338


**Stress findings**
- Both stress scenarios increase mean PD and tail PD monotonically (severe > mild), as expected.
- EL proxy also increases, indicating higher expected portfolio loss under stressed conditions.
- The LGD proxy average is very high in this dataset; this is consistent with low recoveries for many charged-off loans and should be interpreted as a dataset-specific proxy, not a calibrated IFRS9 LGD model.

## LGD/EAD proxy sanity checks

We summarize:
- default count
- EAD proxy distribution
- recoveries distribution
- LGD proxy distribution (defaults only)

This supports interpretability of EL proxy results.

In [10]:
default_mask = oot_with_proxies[TARGET_COL] == 1
print("Defaults in OOT:", int(default_mask.sum()), "out of", len(oot_with_proxies))

display(oot_with_proxies["EAD_proxy"].describe())
display(oot_with_proxies.loc[default_mask, "recoveries"].describe())
display(oot_with_proxies.loc[default_mask, "LGD_proxy"].describe())

Defaults in OOT: 6951 out of 47191


count    47191.000000
mean      2341.710530
std       6806.596234
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      40000.000000
Name: EAD_proxy, dtype: float64

count     6951.000000
mean       332.389835
std       1136.997630
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      33122.070000
Name: recoveries, dtype: float64

count    6951.000000
mean        0.979338
std         0.063286
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: LGD_proxy, dtype: float64

In [11]:
# LGD by grade (report-ready table)
lgd_by_grade = (
    oot_with_proxies.loc[default_mask]
    .groupby("grade")["LGD_proxy"]
    .agg(["count", "mean", "median", lambda x: np.quantile(x, 0.9)])
    .rename(columns={"<lambda_0>": "p90"})
    .sort_values("count", ascending=False)
)
lgd_by_grade

,count,mean,median,p90
grade,,,,
C,2128,0.980893,1.0,1.0
D,1974,0.977382,1.0,1.0
B,1402,0.981952,1.0,1.0
E,707,0.974824,1.0,1.0
A,519,0.983331,1.0,1.0
F,175,0.971131,1.0,1.0
G,46,0.967148,1.0,1.0


**LGD proxy note**
LGD proxy is concentrated near 1.0 across grades (median and p90 ~ 1.0), implying recoveries are typically very small relative to the EAD proxy.
This is plausible for unsecured consumer charge-offs, but it also highlights that this LGD is a coarse proxy for demonstration purposes.

# Model Risk Assessment (Champion)

## Overall conclusion
**Proposed rating: MODERATE Model Risk**

The Champion model shows reasonable discrimination and strong in-sample calibration, but exhibits **material drift** in key inputs and **calibration degradation OOT**, requiring monitoring and potential recalibration.

## Evidence

### 1) Business relevance (segment performance)
- High-volume segments (grades A/B/C; common purposes) show moderate discrimination.
- Portfolio composition changes in OOT (e.g., fewer extreme grades), indicating **mix drift**.
- `term=60 months` appears weaker OOT than `36 months`, suggesting a segment to monitor.

### 2) Stability & drift (PSI/KS)
- PSI highlights key drift drivers:
  - `revol_util` (RED) is the strongest drift feature.
  - `application_type` (high AMBER) and `int_rate` (AMBER) also drift.
- Score distribution shift is measurable (KS ~ 0.058), consistent with feature drift.

### 3) Robustness (sensitivity)
- Ranking stability under ±5%/±10% shocks is extremely high (Spearman ~ 1).
- `dti` is the most sensitivity-relevant driver among tested variables.
- Drift risk (PSI) and local sensitivity are distinct; a feature may drift while small perturbations still produce stable rankings.

### 4) Stress and loss proxy (PD × LGD × EAD)
- Stress scenarios increase mean PD and tail PD monotonically (severe > mild).
- EL proxy increases under stress, supporting vulnerability assessment.
- LGD proxy is high in this dataset due to low recoveries; interpret as a proxy, not a calibrated LGD model.

## Recommended actions (monitoring & remediation)
1) **Monitoring (quarterly/monthly)**
   - Track PSI for `revol_util`, `application_type`, `int_rate`, and score (`pd_pred`).
   - Set investigation triggers: AMBER/RED thresholds.
   - Track missingness drift for key fields (e.g., employment length).

2) **Calibration governance**
   - Monitor calibration-in-the-large and slope OOT.
   - Apply periodic recalibration if PD levels deviate materially from observed default rates.

3) **Segment controls**
   - Monitor performance and calibration specifically for `term=60 months` and high-volume purpose categories.

4) **Documentation**
   - Record assumptions of EAD/LGD proxies and limitations of the EL proxy for transparency.

## Export artifacts (tables)
We export monitoring-ready tables to `reports/tables/` for reuse in the final PDF report.

In [12]:
from pathlib import Path

out_dir = PROJECT_ROOT / "reports" / "tables"
out_dir.mkdir(parents=True, exist_ok=True)

psi_table.to_csv(out_dir / "psi_table.csv", index=False)
sens_table.to_csv(out_dir / "sensitivity_table.csv", index=False)
stress_table.to_csv(out_dir / "stress_table.csv", index=False)
lgd_by_grade.to_csv(out_dir / "lgd_by_grade.csv")

print("Saved tables to:", out_dir)

Saved tables to: /Users/noelp/PycharmProjects/model-risk-validation-pd-lgd-ead/reports/tables
